# Self-attention, step by step

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astudentfromsustech/agents-course/blob/main/practice/p0_attention/walkthrough.ipynb)

*Runs in the browser with nothing installed — or locally with `jupyter lab`.*

Run each cell, look at the numbers, then **change them and run again**. That is the
point of this notebook — the shapes are deliberately tiny (4 tokens, `d_model=6`,
2 heads) so every matrix prints in full and you can check the arithmetic by hand.

Three cells ask you to verify a number yourself before the notebook confirms it.
Do at least one. It is the difference between reading and knowing.

Companion to chapter 01 of the course. Needs only `numpy`.

In [ ]:
import sys
print("running on:", "Colab" if "google.colab" in sys.modules else "local Jupyter",
      "— this lab is pure numpy, so the hardware does not matter")

import numpy as np
np.set_printoptions(precision=2, suppress=True, linewidth=120)

TOK = ["The", "cat", "sat", "down"]
n, d_model, H = 4, 6, 2
d_head = d_model // H

def show(name, M, note=""):
    print(f"{name}   shape {M.shape}" + (f"   {note}" if note else ""))
    for i, row in enumerate(np.atleast_2d(M)):
        label = f"{TOK[i]:>5} |" if M.shape[0] == n else "      |"
        print(f"  {label} {row}")

## 1 · The input

Four tokens, already vectors — embedding happened before attention. Attention never
sees words; it sees these rows and asks *which rows should influence which*.

In [ ]:
rng = np.random.default_rng(1)
X = np.round(rng.normal(size=(n, d_model)), 1)
show("X  (input)", X, "one row per token")

## 2 · The three weight matrices

The only parameters in attention. The **same** matrices are applied to every token — attention has
no per-position weights, which is why it handles any sequence length.

Reading the next line piece by piece, because four separate decisions are packed into it:

| | |
|---|---|
| `default_rng(2)` | A **seeded** random generator, so you and I get the same matrices and the numbers below are reproducible. The `2` is arbitrary — a label picking one stream out of many. Change it to `7` and every number changes; nothing you learn here does. Each cell re-seeds, so cells give the same result whatever order you run them in. |
| `rng.normal(...)` | In a real transformer these are **learned by gradient descent**. Random values stand in for "some trained model" — we care about the *mechanism*, not what a particular model learned. |
| `size=(d_model, d_model)` | The projection maps a token vector to another vector of the same width: `X (4,6) @ W (6,6) → (4,6)`. Square because query, key and value all stay at `d_model` here. (Real models often project to a smaller width and make up the difference with more heads.) |
| `* 0.5` | Shrinks the weights. Real initialisers scale by roughly `1/√d_model` for the same reason — big weights make big dot products, and §5 shows what big dot products do to softmax. Set it to `3.0` and watch step 7 collapse onto one token. |
| `np.round(..., 1)` | **Cosmetic only.** One decimal place so the matrices print readably and you can check the arithmetic by hand. No real model does this. |

**Why a generator expression rather than one matrix?** `for _ in range(3)` draws **three different**
matrices from the same stream. The tempting shortcut `Wq = Wk = Wv = ...` would make all three
*identical* — and then every token's query would be its own key, so each token would mostly attend
to itself. Query, key and value have to be different projections or the mechanism collapses.

> **Try:** set `scale = 3.0` below and look at step 7. Then set `Wk = Wq` and look again.

In [ ]:
rng = np.random.default_rng(2)   # seeded -> reproducible; the 2 is just a label
scale = 0.5                      # try 3.0 and watch step 7 collapse onto one token

# three SEPARATE draws -- Wq = Wk = Wv would make every query match its own key
Wq, Wk, Wv = (np.round(rng.normal(size=(d_model, d_model)) * scale, 1) for _ in range(3))

show("W_q", Wq)
print(f"\nthree different matrices? W_q == W_k: {np.array_equal(Wq, Wk)}")

## 3 · Project X into Q, K, V

Each token produces three vectors **from its own row only**:

| | |
|---|---|
| **query** | what am I looking for? |
| **key** | what do I advertise? |
| **value** | what do I pass on if chosen? |

### What `@` means

`@` is Python's **matrix multiplication** operator (`np.matmul`). It is not `*`:

| | |
|---|---|
| `A * B` | **elementwise** — multiply matching positions. Shapes must match |
| `A @ B` | **matrix product** — row *i* of `A` dotted against every column of `B` |

`X @ Wq` is `(4,6) @ (6,6) → (4,6)`: the inner dimensions must agree (6 and 6), and the
result keeps the outer ones. Four tokens in, four queries out.

**The part worth pausing on:** row *i* of the result depends on row *i* of `X` and nothing else.
Token 2's query is built only from token 2. No mixing happens here — that is why the *same* three
matrices can serve every position, and why a transformer handles any sequence length. The mixing
comes later, in §4, when queries meet keys.

In [ ]:
Q, K, V = X @ Wq, X @ Wk, X @ Wv
show("Q", Q); print(); show("K", K); print(); show("V", V)

# @ is matrix product, * is elementwise -- they are completely different operations
print("\nX @ Wq  ->", (X @ Wq).shape, " matrix product")
print("X * X   ->", (X * X).shape, " elementwise, same shape in and out")

# row i of Q comes ONLY from row i of X
print(f"\nQ[1] built from X[1] alone?  {np.allclose(X[1] @ Wq, Q[1])}")

# and the inner dimensions must agree: (a,b) @ (b,c) -> (a,c)
try:
    X @ X
    print("X @ X worked?!")
except ValueError:
    print(f"\nX @ X fails: {X.shape} @ {X.shape} -- the inner dims (6 and 4) disagree")
print(f"X.T @ X works: {X.T.shape} @ {X.shape} -> {(X.T @ X).shape}  (inner dims both 4)")

### Why three projections, instead of just using X?

The names *query*, *key*, *value* describe what the vectors do, not why they must exist. Three
questions, each answered by one of the three matrices.

**Why project at all?** Suppose you skipped it and scored tokens directly: `X @ X.T`. Relevance
would then be *embedding similarity*, fixed forever — "cat" attends to "dog" because their
embeddings are close, and no amount of training changes what relevance means. The projections make
relevance **learnable**: `W_q` and `W_k` define a space in which closeness means whatever the task
needs it to mean, which may have nothing to do with the words being similar.

**Why are `W_q` and `W_k` different?** This is the one that matters. If they were the same matrix,
the score matrix would be `(XW)(XW)ᵀ` — **symmetric**. Token *i* would attend to *j* exactly as
much as *j* attends to *i*. But relevance is directional: an adjective looks for the noun it
modifies; the noun is not equally looking for the adjective. A pronoun looks back for its
antecedent, not the other way round. **Two different matrices are what buy you a one-way relation.**

**Why is `V` separate from `K`?** Because *what makes me findable* and *what I contribute* are
different things. The key is an address; the value is the payload. Tying them would force a token
to hand over the very same vector that got it selected.

It is a soft dictionary lookup. `d[k]` returns the one value whose key matches exactly; attention
returns a weighted blend over *every* value, weighted by how well each key matches the query.

The next cell checks the symmetry claim rather than asking you to believe it.

In [ ]:
sym = lambda M: np.allclose(M, M.T)

print("no projection    scores = X @ X.T          symmetric?", sym(X @ X.T))
print("tied  W_q = W_k  scores = (XWq)(XWq).T     symmetric?", sym((X @ Wq) @ (X @ Wq).T))
print("separate         scores = (XWq)(XWk).T     symmetric?", sym(Q @ K.T))

# what symmetry costs you, concretely
S_tied = (X @ Wq) @ (X @ Wq).T
S_real = Q @ K.T
print(f"\ntied:     'cat'->'The' {S_tied[1,0]:6.2f}   'The'->'cat' {S_tied[0,1]:6.2f}   forced equal")
print(f"separate: 'cat'->'The' {S_real[1,0]:6.2f}   'The'->'cat' {S_real[0,1]:6.2f}   free to differ")

# The projections ARE a learned bilinear form. Note the row-vector convention here:
# X is (n, d) with tokens as ROWS, so scores = (X Wq)(X Wk).T = X (Wq Wk.T) X.T
M = Wq @ Wk.T
print(f"\nscores == X @ M @ X.T  with M = Wq @ Wk.T ?  {np.allclose(Q @ K.T, X @ M @ X.T)}")
print(f"  M is {M.shape} = d_model x d_model, but rank <= d_k")
print(f"  rank(M) = {np.linalg.matrix_rank(M)}   (full here only because d_k == d_model in this toy)")
print(f"  M symmetric? {np.allclose(M, M.T)}   <- asymmetry is what makes relevance directional")
print(f"  raw dot product is the M = I case: {np.allclose(X @ np.eye(d_model) @ X.T, X @ X.T)}")

## 4 · Scores — every query against every key

`scores[i,j]` is how much token *i* wants token *j*.

**This is the only place information moves between tokens.** Everything else in a
transformer processes each position alone.

In [ ]:
scores = Q @ K.T
show("scores = Q @ K.T", scores, "rows = queries, cols = keys")

# ✋ CHECK: scores[1,0] is the dot product of query 'cat' with key 'The'
hand = float(np.dot(Q[1], K[0]))
print(f"\nby hand -> {hand:.2f}    from matrix -> {scores[1,0]:.2f}   match: {np.isclose(hand, scores[1,0])}")

## 5 · Scale by √d_k

Harmless at this size. At `d_k = 256` the unscaled version saturates softmax into a
hard argmax — `attention_lab.py` step 2 measures that collapse directly.

> **Try:** set `scale = 1.0` (no scaling) and look at how much sharper step 7 becomes.

In [ ]:
scale = np.sqrt(d_model)
scaled = scores / scale
print(f"max|score|  before {np.abs(scores).max():.2f}   after {np.abs(scaled).max():.2f}")
show("scaled", scaled)

## 6 · Mask the future

Token *i* may not look at anything after *i*. `-inf` becomes exactly 0 after softmax.

> **Try:** comment out the mask and watch row `The` in step 7 stop being `[1,0,0,0]`.

In [ ]:
mask = np.tril(np.ones((n, n), dtype=bool))
masked = np.where(mask, scaled, -np.inf)
show("mask (1 = allowed)", mask.astype(int)); print()
show("scores after masking", masked)

## 7 · Softmax → attention weights

Scores are arbitrary real numbers — some negative, some large. They cannot be used as weights
directly. Softmax turns each **row** into a probability distribution:

$$\text{softmax}(s)_j = \frac{e^{s_j}}{\sum_k e^{s_k}}$$

Two steps, each doing a specific job:

| | |
|---|---|
| `exp` | makes every value **positive** (so negative scores become small weights, not negative ones) and turns differences into **ratios** — a score 1 higher earns `e ≈ 2.72×` more weight, 2 higher earns `7.4×`. This is what makes it a *soft* argmax rather than an average |
| divide by the sum | normalises to 1, so each row is a genuine distribution over the keys |

Three properties worth knowing, all checked in the next cell:

- **Shift-invariant.** `softmax(s) == softmax(s + c)` for any constant `c`. Adding the same number
  to every score changes nothing — which is exactly why subtracting the row max is free.
- **Not scale-invariant.** `softmax(2s)` is *sharper* than `softmax(s)`. Multiplying the scores
  concentrates the weight. **This is the whole `√d_k` story from §5** — dividing the scores is
  deliberately flattening the distribution.
- **`exp(-inf) = 0` exactly.** That is why masking is done on the *scores*, before softmax, and not
  on the weights after: it produces a true zero and the remaining weights still sum to 1.

🔴 **Rows sum to 1; columns do not.** Each query spends its attention budget across the keys. A key
can be ignored by everyone or used by everyone — nothing normalises down a column.

### Why subtract the max

`exp` overflows fast — `exp(1000)` is `inf`, and `inf/inf` is `nan`. Subtracting the row max puts
the largest value at `exp(0) = 1` and everything else below it, so the computation stays in range.
By shift-invariance the answer is unchanged. Every real implementation does this.

In [ ]:
e = np.exp(masked - masked.max(-1, keepdims=True))   # the -max is the stability trick
A = e / e.sum(-1, keepdims=True)
show("A  (attention weights)", A)

print(f"\nrow sums    -> {A.sum(1)}   each query spends a budget of 1")
print(f"column sums -> {np.round(A.sum(0), 2)}   NOT 1, and nothing says they should be")

# shift-invariant: adding a constant changes nothing
sm = lambda s: np.exp(s - s.max(-1, keepdims=True)) / np.exp(s - s.max(-1, keepdims=True)).sum(-1, keepdims=True)
print(f"\nsoftmax(s) == softmax(s + 100)?  {np.allclose(sm(scaled), sm(scaled + 100))}   shift-invariant")

# but NOT scale-invariant -- this is the sqrt(d_k) story
for f in (0.25, 1, 4):
    print(f"  softmax({f:>4} x s)  peak weight {sm(scaled * f).max():.3f}   "
          f"{'flatter' if f < 1 else ('as computed' if f == 1 else 'sharper -- toward argmax')}")

# why the -max matters at all
big = np.array([[1000.0, 1001.0]])
with np.errstate(over="ignore", invalid="ignore"):
    naive = np.exp(big) / np.exp(big).sum(-1, keepdims=True)
print(f"\nwithout the -max: exp(1000)/sum -> {naive}   (inf/inf)")
print(f"with    the -max:                 -> {sm(big)}   correct")

print(f"\nmasked entries are exactly zero? {np.all(A[np.triu_indices(n, 1)] == 0)}   because exp(-inf) = 0")

## 8 · Output — the weighted sum

Each output row is a weighted average of the **value** rows, using that row's weights.
That is self-attention, complete.

In [ ]:
out = A @ V
show("output = A @ V", out)

# ✋ CHECK: build row 'sat' by hand as a weighted sum of V
hand = sum(A[2, j] * V[j] for j in range(n))
for j in range(3):
    print(f"   {A[2,j]:.2f} x V[{TOK[j]}]")
print(f"\nby hand  -> {np.round(hand,2)}")
print(f"from A@V -> {np.round(out[2],2)}   match: {np.allclose(hand, out[2])}")

## 9 · Multi-head — split, attend separately

Same Q, K, V — now cut into 2 slices of width 3, each attended **independently**.

Head 0's queries never touch head 1's keys. That separation is the whole of multi-head,
and it is what `attention_lab.py` tests by perturbing one head and demanding the others
come out bit-identical.

> **Try:** set `H = 3` at the top (6 ÷ 3 = 2-wide heads) and re-run from cell 1.
> More heads, narrower each. What happens to how sharp each head's attention is?

In [ ]:
As, outs = [], []
for h in range(H):
    sl = slice(h * d_head, (h + 1) * d_head)
    sc = Q[:, sl] @ K[:, sl].T / np.sqrt(d_head)
    sc = np.where(mask, sc, -np.inf)
    ee = np.exp(sc - sc.max(-1, keepdims=True))
    a = ee / ee.sum(-1, keepdims=True)
    As.append(a); outs.append(a @ V[:, sl])
    show(f"A (head {h})", a); print()

print("Two different opinions about relevance, from the same input.")

## 10 · Concatenate and project

`W_o` is where the heads finally mix — and the **only** place they do.

That is one attention layer, end to end. A transformer stacks this with a feed-forward
block and a residual add, N times. Nothing else is new.

In [ ]:
cat = np.concatenate(outs, axis=-1)
rng = np.random.default_rng(3)
Wo = np.round(rng.normal(size=(d_model, d_model)) * 0.5, 1)
show("concat(head0, head1)", cat, f"{d_head} + {d_head} = {d_model}")
print(); show("final = concat @ W_o", cat @ Wo)

## 11 · So what is the output actually *for*?

You now have one output vector per token. **What you bought is context.**

Before attention, a token's vector is the same wherever it appears — "bank" by a river and "bank"
holding money are one vector. After attention, each token's vector has been mixed with whatever it
attended to, so the same input token in two different sentences comes out **different**. That is
the product: *contextualised* representations.

The cell below proves it — the same token, identical input row, two different neighbourhoods.

In [ ]:
def attend(Xin):
    q, k, v = Xin @ Wq, Xin @ Wk, Xin @ Wv
    s = np.where(mask, q @ k.T / np.sqrt(d_model), -np.inf)
    e = np.exp(s - s.max(-1, keepdims=True))
    return (e / e.sum(-1, keepdims=True)) @ v

A_ctx, B_ctx = X.copy(), X.copy()
B_ctx[0] = B_ctx[2] = -X[0]          # same token 1, different neighbours around it

print("token 1's INPUT row identical in both? ", np.allclose(A_ctx[1], B_ctx[1]))
oa, ob = attend(A_ctx)[1], attend(B_ctx)[1]
print("token 1's OUTPUT row identical?        ", np.allclose(oa, ob))
print(f"\n  context A -> {np.round(oa, 2)}")
print(f"  context B -> {np.round(ob, 2)}")
print(f"  they differ by {np.abs(oa - ob).max():.2f} at the largest coordinate")

### What gets bolted on next

The contextualised vectors are an intermediate. What you attach decides the task — and attention
itself does not change:

| task | what you attach |
|---|---|
| **generate text** | take the **last** position's vector → linear → logits over the vocabulary → sample. That is GPT, and it is the loop in [chapter 03](https://astudentfromsustech.github.io/agents-course/03_agent/03_agent.html) |
| **classify a sequence** | pool all positions (or use one designated token) → linear → classes |
| **label every token** | each position → linear → tag. NER, part-of-speech, secondary structure |
| **embed for retrieval** | pool → one vector, compared by cosine similarity. This is what [chapter 05](https://astudentfromsustech.github.io/agents-course/05_knowledge/05_knowledge.html) means by "dense retrieval" |
| **encoder → decoder** | the encoder's outputs become the **K and V** that a decoder cross-attends to |

None of this is text-specific. Swap tokens for image patches and you get a vision transformer; for
amino acids, a protein language model; for genomic bins, the sequence models used for regulatory
prediction. **Attention only needs a set of vectors and a reason to relate them.**

### And why stack it N times

One layer gives each token one round of looking around. Stacking lets information travel in
**hops**: layer 2 can use what layer 1 gathered, which no single layer could reach.

You do not have to take that on faith — `p1` shows the mechanism forming. A **previous-token head**
in layer 0 writes "what came before me" into each position; an **induction head** in layer 1 reads
*that* and matches on it. Neither half works alone, and one layer cannot do the job. That two-layer
composition is the whole reason for depth.

---

## Now write it yourself

This notebook showed you the mechanism. `attention_lab.py` in the same folder makes you
**build** it — three `TODO`s, checked against a longhand reference loop, with behavioural
tests for the two properties that matter (heads don't mix; masking makes the past
independent of the future).

```bash
python3 attention_lab.py
```

Then `p1_induction_heads/` trains a tiny transformer until it discovers an induction
circuit, and scores every head against a positional control.